<a target="_blank" href="../cluster" style="font-size:20px">All Applications (YARN)</a>

# Стек Hadoop. Практическая работа

## Цель практической работы

Научиться использовать Hadoop MapReduce на практике.

## Что входит в работу

* Загрузка данных в HDFS.
* Получение данных из HDFS.
* Реализация парадигмы MapReduce с применением Hadoop Streaming.

## Формат сдачи

Отправьте в форме сдачи следующие файлы:
- файл с результатом result.json;
- ноутбук с кодом (все команды и функции, которые использовались для решения задач).

# Практическое задание

Будем использовать логи сессий прослушивания музыкальных исполнителей в сервисе Spotify, сокращённую версию.

https://www.aicrowd.com/challenges/spotify-sequential-skip-prediction-challenge/dataset_files

Файл `spotify/log_mini.csv` содержит записи вида `ID сессии, номер в сессии, длинна сессии, id трека, skip_1, skip_2, ...`:
```csv
session_id,session_position,session_length,track_id_clean,skip_1,skip_2,skip_3,not_skipped,context_switch,no_pause_before_play,short_pause_before_play,long_pause_before_play,hist_user_behavior_n_seekfwd,hist_user_behavior_n_seekback,hist_user_behavior_is_shuffle,hour_of_day,date,premium,context_type,hist_user_behavior_reason_start,hist_user_behavior_reason_end
0_00006f66-33e5-4de7-a324-2d18e439fc1e,1,20,t_0479f24c-27d2-46d6-a00c-7ec928f2b539,false,false,false,true,0,0,0,0,0,0,true,16,2018-07-15,true,editorial_playlist,trackdone,trackdone
0_00006f66-33e5-4de7-a324-2d18e439fc1e,2,20,t_9099cd7b-c238-47b7-9381-f23f2c1d1043,false,false,false,true,0,1,0,0,0,0,true,16,2018-07-15,true,editorial_playlist,trackdone,trackdone
```

Вам нужно:
1. **Посчитать для каждого трека количество его прослушиваний. Выведите два самых прослушиваемых трека.**
2. **Вывести долю популярных треков: тех, что имеют больше 100 прослушиваний.**

Для решения задачи:
1. Скопируйте файлы в HDFS.
2. Реализуйте подсчёт прослушиваний отдельным MapReduce, в файлы результата сохраните пары <track_id, listen_count>.
3. С помощью команды `hdfs dfs -cat <YOUR-MAPRED-RESULT/*> | python stream_processor.py` решите три подзадачи:
    1. Подсчитайте количество уникальных треков.
    2. Посчитайте количество треков с количеством прослушиваний больше 20.
    3. Найдите два самых популярных по listen_count.
    
    `stream_processor.py` — скрипт, читающий с потока ввода, необходимо реализовать самостоятельно.
4. Сохраните результат работы скрипта выше в файл `result.json`, формат описан ниже.

Реализуйте решение с использованием Hadoop MapReduce Streaming, для написания mapper и reducer используйте Python.

Решение сохраните в локальный файл `result.json`, где по ключу q1
 запишите ответ на первый вопрос, по ключу q2 — на второй и по ключу q3 — на третий.


## Критерии проверки

1. Корректно реализован алгоритм подсчёта прослушиваний — mapper.py, reducer.py (без сохранения всех данных в память, работа с потоком).
2. mapper.py и reducer.py протестированы локально.
3. Данные ( `spotify/log_mini.csv` ) загружены в HDFS.
4. Корректно запущен процесс Hadoop MapReduce Streaming с использованием mapper.py и reducer.py на данных.
5. Корректно реализован `stream_processor.py` (без сохранения всех данных в память, работа с потоком).
6. Результат записан в файл `result.json` и совпадает с эталонным.

Пример содержимого файла `result.json`:

```json
{
    "q1": ["id1", "id2"],
    "q2": 0.13
}
```

In [ ]:
# Пример содержимого файла
! head -n 5 spotify/log_mini.csv

In [ ]:
# Копируем файлы в HDFS
! hadoop fs -copyFromLocal ... ... # допишите команду

In [ ]:
%%writefile mapper.py
# Реализуйте mapper
import sys

for line in sys.stdin:
    # line - строки из файла spotify/log_mini.csv
    # Ваш код здесь, обработайте line
    # ...
    print(...) # Выведите строки на поток вывода: <track_id>\t1

In [ ]:
# Протестируйте mapper локально
! head -n 5 spotify/log_mini.csv | python mapper.py

In [ ]:
%%writefile reducer.py
# Реализуйте reducer
import sys

for line in sys.stdin:
    # line - группа строк из выхода mapper.py
    # Ваш код здесь, обработайте line
    # ...
    print(...) # Выведите строки на поток вывода

In [ ]:
# Протестируйте reducer локально
! python -c "print('\n'.join([f'{x}\t1' for x in (['aa'] * 10 + ['bb'] * 5)]))" | python reducer.py

In [ ]:
# Запустите MapReduce Streaming
! mapred streaming \
  -input /spotify/log_mini.csv \
  -output /track-count \
  -mapper "/opt/conda/bin/python mapper.py" \
  -reducer "/opt/conda/bin/python reducer.py" \
  -file mapper.py \
  -file reducer.py

In [ ]:
%%writefile stream_processor.py
# Реализуйте код обработки результата MapReduce
import sys

# Подзадачи:
# 1. Подсчитайте количество уникальных треков:
uniq_tracks_count = 0
# 2. Посчитайте количество треков с количеством прослушиваний больше 20:
popular_tracks_count = 0
# 3. Найдите два самых популярных по listen_count:
top_2_tracks = [None, None]

for line in sys.stdin:
    # line - результат работы MapReduce, финальный формат от reducer.py - <track_id>\t<N>
    # ваш код здесь
    # ...

data = {
    'q1': uniq_tracks_count,
    'q2': popular_tracks_count,
    'q3': top_2_tracks,
}
with open('result.json', 'w') as f:
    f.write(json.dumps(data))


In [ ]:
# Обработайте данные из HDFS с помощью stream_processor.py
! hdfs dfs -cat /track-count/* | python stream_processor.py

In [ ]:
# Выведите содержимое файла result.json
! cat result.json